In [1]:
%pip install duckdb

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import duckdb

print(duckdb.__version__)

2.0.0


In [3]:
import duckdb

parquet_path = r"D:\Data Analysis Projects_\favorita-grocery-sales-forecasting\Notebook\favorita_exports\Parquet\stores_pl.parquet"

con = duckdb.connect(
    r"D:\Data Analysis Projects_\favorita-grocery-sales-forecasting\Notebook\favorita.duckdb"
)

con.execute(
    "SELECT * FROM read_parquet(?) LIMIT 5",
    [parquet_path]
).fetchdf()

,store_nbr,city,state,type
0,1,Quito,Pichincha,D
1,2,Quito,Pichincha,D
2,3,Quito,Pichincha,D
3,4,Quito,Pichincha,D
4,5,Santo Domingo,Santo Domingo de los Tsachilas,D


In [4]:
con.execute("""
    CREATE OR REPLACE TABLE stores AS
    SELECT *
    FROM read_parquet(?)
""", [parquet_path])

In [5]:
con.execute("SELECT * FROM stores LIMIT 5").fetchdf()

,store_nbr,city,state,type
0,1,Quito,Pichincha,D
1,2,Quito,Pichincha,D
2,3,Quito,Pichincha,D
3,4,Quito,Pichincha,D
4,5,Santo Domingo,Santo Domingo de los Tsachilas,D


In [6]:
from pathlib import Path

export_dir = Path(
    r"D:\Data Analysis Projects_\favorita-grocery-sales-forecasting\Notebook\favorita_exports\Parquet"
)

files = {
    "items": "items_pl.parquet",
    "transactions": "transactions_pl.parquet",
    "daily_store_sales": "daily_store_sales.parquet",
}

for table_name, filename in files.items():
    file_path = export_dir / filename

    con.execute(
        f"""
        CREATE OR REPLACE TABLE {table_name} AS
        SELECT *
        FROM read_parquet(?)
        """,
        [str(file_path)]
    )

    print(f"{table_name}: created successfully")

items: created successfully
transactions: created successfully
daily_store_sales: created successfully


In [7]:
tables = [
    "stores",
    "items",
    "transactions",
    "daily_store_sales"
]

for table in tables:
    result = con.execute(f"""
        SELECT
            COUNT(*) AS row_count
        FROM {table}
    """).fetchone()

    columns = con.execute(
        f"DESCRIBE {table}"
    ).fetchdf()

    print(f"\nTable: {table}")
    print(f"Rows: {result[0]}")
    print(f"Columns: {len(columns)}")
    print(columns[["column_name", "column_type"]])


Table: stores
Rows: 54
Columns: 4
  column_name column_type
0   store_nbr      BIGINT
1        city     VARCHAR
2       state     VARCHAR
3        type     VARCHAR

Table: items
Rows: 4100
Columns: 4
  column_name column_type
0    item_nbr      BIGINT
1      family     VARCHAR
2       class      BIGINT
3  perishable      BIGINT

Table: transactions
Rows: 83488
Columns: 3
    column_name column_type
0          date        DATE
1     store_nbr      BIGINT
2  transactions      BIGINT

Table: daily_store_sales
Rows: 83606
Columns: 3
        column_name column_type
0              date        DATE
1         store_nbr      BIGINT
2  daily_unit_sales      DOUBLE


In [8]:
for table in ["stores", "items", "transactions", "daily_store_sales"]:
    row_count = con.execute(
        f"SELECT COUNT(*) FROM {table}"
    ).fetchone()[0]

    print(f"{table}: {row_count:,} rows")

train_path = export_dir / "train_df.parquet"

print(f"\ntrain_df.parquet exists: {train_path.exists()}")
if train_path.exists():
    print(f"File size: {train_path.stat().st_size / (1024**3):.2f} GB")

stores: 54 rows
items: 4,100 rows
transactions: 83,488 rows
daily_store_sales: 83,606 rows

train_df.parquet exists: True
File size: 0.38 GB


In [9]:
train_path = export_dir / "train_df.parquet"

con.execute("""
    SELECT
        COUNT(*) AS row_count,
        MIN(date) AS min_date,
        MAX(date) AS max_date,
        COUNT(DISTINCT store_nbr) AS stores_count,
        COUNT(DISTINCT item_nbr) AS items_count
    FROM read_parquet(?)
""", [str(train_path)]).fetchdf()

,row_count,min_date,max_date,stores_count,items_count
0,125497040,2013-01-01,2017-08-15,54,4036


In [10]:
con.execute("""
    SELECT
        SUM(
            COALESCE(LENGTH(CAST(id AS VARCHAR)), 0) +
            COALESCE(LENGTH(CAST(date AS VARCHAR)), 0) +
            COALESCE(LENGTH(CAST(store_nbr AS VARCHAR)), 0) +
            COALESCE(LENGTH(CAST(item_nbr AS VARCHAR)), 0) +
            COALESCE(LENGTH(CAST(unit_sales AS VARCHAR)), 0) +
            COALESCE(LENGTH(CAST(onpromotion AS VARCHAR)), 0)
        ) AS estimated_text_bytes
    FROM read_parquet(?)
""", [str(train_path)]).fetchdf()

,estimated_text_bytes
0,4.244470e+09


In [11]:
con.execute(f"""
    CREATE OR REPLACE VIEW train AS
    SELECT *
    FROM read_parquet('{train_path.as_posix()}')
""")

In [12]:
con.execute("""
    SELECT
        COUNT(*) AS row_count,
        SUM(unit_sales) AS total_unit_sales
    FROM train
""").fetchdf()

,row_count,total_unit_sales
0,125497040,1.073610e+09


In [13]:
con.close()

con = duckdb.connect(
    r"D:\Data Analysis Projects_\favorita-grocery-sales-forecasting\Notebook\favorita.duckdb"
)

In [14]:
con.execute("""
    SELECT COUNT(*) AS row_count
    FROM train
""").fetchdf()

,row_count
0,125497040


In [15]:
con.execute("SHOW TABLES").fetchdf()

,name
0,bottom_10_items_details
1,daily_store_family_sales
2,daily_store_sales
3,family_sales
4,holiday_weekday_effect
5,items
6,monthly_normalized
7,monthly_store_family_sales
8,oil_sales_analysis
9,promotion_sales


In [16]:
con.execute("""
    SELECT
        YEAR(date) AS year,
        SUM(unit_sales) AS total_unit_sales
    FROM train
    GROUP BY YEAR(date)
    ORDER BY year
""").fetchdf()

,year,total_unit_sales
0,2013,1.404180e+08
1,2014,2.094741e+08
2,2015,2.408678e+08
3,2016,2.886334e+08
4,2017,1.942169e+08


In [17]:
con.execute("""
    SELECT
        YEAR(date) AS year,
        SUM(unit_sales) AS total_unit_sales
    FROM train
    WHERE date >= DATE '2016-01-01'
      AND date <= DATE '2017-08-15'
      AND (
          (YEAR(date) = 2016 AND date <= DATE '2016-08-15')
          OR YEAR(date) = 2017
      )
    GROUP BY YEAR(date)
    ORDER BY year
""").fetchdf()

,year,total_unit_sales
0,2016,1.765439e+08
1,2017,1.942169e+08


In [18]:
con.execute("""
    CREATE OR REPLACE TABLE yearly_sales AS
    SELECT
        YEAR(date) AS year,
        SUM(unit_sales) AS total_unit_sales
    FROM train
    GROUP BY YEAR(date)
    ORDER BY year
""")

In [19]:
con.execute("""
    SELECT *
    FROM yearly_sales
    ORDER BY year
""").fetchdf()

,year,total_unit_sales
0,2013,1.404180e+08
1,2014,2.094741e+08
2,2015,2.408678e+08
3,2016,2.886334e+08
4,2017,1.942169e+08


In [20]:
con.execute("""
    SELECT
        YEAR(date) AS year,
        MONTH(date) AS month,
        SUM(unit_sales) AS total_unit_sales,
        COUNT(DISTINCT date) AS observed_days,
        DAY(LAST_DAY(MIN(date))) AS calendar_days,
        SUM(unit_sales) / COUNT(DISTINCT date)
            AS avg_per_observed_day,
        SUM(unit_sales) / DAY(LAST_DAY(MIN(date)))
            AS avg_per_calendar_day
    FROM train
    GROUP BY
        YEAR(date),
        MONTH(date)
    ORDER BY
        year,
        month
""").fetchdf()

,year,month,total_unit_sales,observed_days,calendar_days,avg_per_observed_day,avg_per_calendar_day
0,2013,1,1.032741e+07,31,31,333142.314065,333142.314065
1,2013,2,9.658960e+06,28,28,344962.849179,344962.849179
2,2013,3,1.142850e+07,31,31,368661.194742,368661.194742
3,2013,4,1.099346e+07,30,30,366448.757967,366448.757967
4,2013,5,1.159770e+07,31,31,374119.419581,374119.419581
5,2013,6,1.168934e+07,30,30,389644.535400,389644.535400
6,2013,7,1.125740e+07,31,31,363141.890548,363141.890548
7,2013,8,1.173779e+07,31,31,378638.320000,378638.320000
8,2013,9,1.179293e+07,30,30,393097.741100,393097.741100
9,2013,10,1.177562e+07,31,31,379858.689032,379858.689032


In [21]:
con.execute("""
    CREATE OR REPLACE TABLE monthly_normalized AS
    SELECT
        YEAR(date) AS year,
        MONTH(date) AS month,
        SUM(unit_sales) AS total_unit_sales,
        COUNT(DISTINCT date) AS observed_days,
        SUM(unit_sales) / COUNT(DISTINCT date)
            AS avg_daily_unit_sales
    FROM train
    GROUP BY
        YEAR(date),
        MONTH(date)
    ORDER BY
        year,
        month
""")

In [22]:
con.execute("""
    (SELECT * FROM monthly_normalized ORDER BY year, month LIMIT 5)
    UNION ALL
    (SELECT * FROM monthly_normalized ORDER BY year DESC, month DESC LIMIT 5)
""").fetchdf()

,year,month,total_unit_sales,observed_days,avg_daily_unit_sales
0,2013,1,1.032741e+07,31,333142.314065
1,2013,2,9.658960e+06,28,344962.849179
2,2013,3,1.142850e+07,31,368661.194742
3,2013,4,1.099346e+07,30,366448.757967
4,2013,5,1.159770e+07,31,374119.419581
5,2017,8,1.243332e+07,15,828888.195933
6,2017,7,2.701147e+07,31,871337.889226
7,2017,6,2.568277e+07,30,856092.270867
8,2017,5,2.691184e+07,31,868123.846774
9,2017,4,2.589531e+07,30,863176.849700


In [23]:
con.execute("""
    CREATE OR REPLACE TABLE weekday_summary AS
    SELECT
        DAYNAME(date) AS day_name,
        DAYOFWEEK(date) AS day_number,
        SUM(unit_sales) AS total_unit_sales,
        COUNT(DISTINCT date) AS observed_days,
        SUM(unit_sales) / COUNT(DISTINCT date)
            AS avg_daily_unit_sales
    FROM train
    GROUP BY
        DAYNAME(date),
        DAYOFWEEK(date)
    ORDER BY
        day_number
""")

In [24]:
con.execute("""
    SELECT *
    FROM weekday_summary
    ORDER BY day_number
""").fetchdf()

,day_name,day_number,total_unit_sales,observed_days,avg_daily_unit_sales
0,Sunday,0,1.980523e+08,240,825218.075771
1,Monday,1,1.488127e+08,241,617479.929934
2,Tuesday,2,1.379198e+08,242,569916.489993
3,Wednesday,3,1.423786e+08,240,593244.190965
4,Thursday,4,1.212616e+08,240,505256.688612
5,Friday,5,1.390978e+08,240,579573.986125
6,Saturday,6,1.860875e+08,241,772147.340838


In [25]:
con.execute("""
    CREATE OR REPLACE TABLE store_total_sales AS
    WITH store_sales AS (
        SELECT
            store_nbr,
            SUM(unit_sales) AS total_unit_sales
        FROM train
        GROUP BY store_nbr
    )
    SELECT
        store_nbr,
        total_unit_sales,
        100.0 * total_unit_sales
            / SUM(total_unit_sales) OVER ()
            AS sales_contribution_pct,
        RANK() OVER (
            ORDER BY total_unit_sales DESC
        ) AS sales_rank
    FROM store_sales
    ORDER BY sales_rank
""")

In [26]:
con.execute("""
    SELECT *
    FROM store_total_sales
    ORDER BY sales_rank
    LIMIT 10
""").fetchdf()

,store_nbr,total_unit_sales,sales_contribution_pct,sales_rank
0,44,6.208754e+07,5.783062,1
1,45,5.449801e+07,5.076145,2
2,47,5.094831e+07,4.745512,3
3,3,5.048190e+07,4.702069,4
4,49,4.342009e+07,4.044306,5
5,46,4.189605e+07,3.902352,6
6,48,3.593313e+07,3.346944,7
7,51,3.291148e+07,3.065496,8
8,8,3.049134e+07,2.840075,9
9,50,2.865302e+07,2.668847,10


In [27]:
con.execute("""
    SELECT
        COUNT(*) AS store_count,
        SUM(total_unit_sales) AS total_unit_sales,
        SUM(sales_contribution_pct) AS total_contribution_pct,
        SUM(
            CASE WHEN sales_rank <= 10
                 THEN sales_contribution_pct
                 ELSE 0
            END
        ) AS top_10_contribution_pct
    FROM store_total_sales
""").fetchdf()

,store_count,total_unit_sales,total_contribution_pct,top_10_contribution_pct
0,54,1.073610e+09,100.0,40.174809


In [28]:
con.execute("""
    CREATE OR REPLACE TABLE store_type_sales AS
    SELECT
        s.type AS store_type,
        COUNT(*) AS store_count,
        SUM(st.total_unit_sales) AS total_unit_sales,
        SUM(st.total_unit_sales) / COUNT(*) AS avg_sales_per_store,
        100.0 * SUM(st.total_unit_sales)
            / SUM(SUM(st.total_unit_sales)) OVER ()
            AS sales_contribution_pct
    FROM store_total_sales st
    JOIN stores s
        ON st.store_nbr = s.store_nbr
    GROUP BY s.type
    ORDER BY total_unit_sales DESC
""")

In [29]:
con.execute("""
    SELECT *
    FROM store_type_sales
    ORDER BY total_unit_sales DESC
""").fetchdf()

,store_type,store_count,total_unit_sales,avg_sales_per_store,sales_contribution_pct
0,A,9,3.530438e+08,3.922709e+07,32.883796
1,D,18,3.510633e+08,1.950352e+07,32.699326
2,C,15,1.644345e+08,1.096230e+07,15.316028
3,B,8,1.452483e+08,1.815603e+07,13.528956
4,E,4,5.982042e+07,1.495511e+07,5.571894


In [30]:
con.execute("""
    CREATE OR REPLACE TABLE store_yearly_sales AS
    SELECT
        store_nbr,
        YEAR(date) AS year,
        SUM(unit_sales) AS total_unit_sales
    FROM train
    GROUP BY
        store_nbr,
        YEAR(date)
    ORDER BY
        store_nbr,
        year
""")

In [31]:
con.execute("""
    SELECT *
    FROM store_yearly_sales
    ORDER BY store_nbr, year
    LIMIT 10
""").fetchdf()

,store_nbr,year,total_unit_sales
0,1,2013,1.894713e+06
1,1,2014,2.899710e+06
2,1,2015,3.120580e+06
3,1,2016,3.793493e+06
4,1,2017,2.436517e+06
5,2,2013,2.865393e+06
6,2,2014,4.426115e+06
7,2,2015,4.815042e+06
8,2,2016,5.883244e+06
9,2,2017,3.564823e+06


In [32]:
con.execute("""
    CREATE OR REPLACE TABLE family_sales AS
    WITH family_totals AS (
        SELECT
            i.family,
            SUM(t.unit_sales) AS total_unit_sales,
            COUNT(DISTINCT t.item_nbr) AS number_of_products
        FROM train t
        JOIN items i
            ON t.item_nbr = i.item_nbr
        GROUP BY i.family
    ),
    family_contributions AS (
        SELECT
            family,
            total_unit_sales,
            100.0 * total_unit_sales
                / SUM(total_unit_sales) OVER ()
                AS sales_contribution_pct,
            number_of_products,
            total_unit_sales / number_of_products
                AS avg_units_per_product
        FROM family_totals
    )
    SELECT
        *,
        SUM(sales_contribution_pct) OVER (
            ORDER BY total_unit_sales DESC
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS cumulative_contribution_pct
    FROM family_contributions
    ORDER BY total_unit_sales DESC
""")

In [33]:
con.execute("""
    SELECT *
    FROM family_sales
    ORDER BY total_unit_sales DESC
""").fetchdf()

,family,total_unit_sales,sales_contribution_pct,number_of_products,avg_units_per_product,cumulative_contribution_pct
0,grocery i,3.434617e+08,31.991286,1290,266249.401462,31.991286
1,beverages,2.169422e+08,20.206797,604,359175.903974,52.198083
2,produce,1.227047e+08,11.429164,306,400995.681955,63.627247
3,cleaning,9.751840e+07,9.083221,446,218651.116592,72.710468
4,dairy,6.448726e+07,6.006580,242,266476.264463,78.717048
5,bread/bakery,4.213395e+07,3.924510,134,314432.429657,82.641558
6,poultry,3.187600e+07,2.969048,54,590296.379102,85.610606
7,meats,3.108647e+07,2.895508,84,370077.004854,88.506114
8,personal care,2.458094e+07,2.289559,153,160659.758170,90.795673
9,deli,2.411032e+07,2.245724,91,264948.598558,93.041397


In [34]:
con.execute("""
    CREATE OR REPLACE TABLE top_10_items_details AS
    SELECT
        t.item_nbr,
        i.family,
        SUM(t.unit_sales) AS total_unit_sales
    FROM train t
    JOIN items i
        ON t.item_nbr = i.item_nbr
    GROUP BY
        t.item_nbr,
        i.family
    ORDER BY total_unit_sales DESC
    LIMIT 10
""")

In [35]:
con.execute("""
    SELECT *
    FROM top_10_items_details
    ORDER BY total_unit_sales DESC
""").fetchdf()

,item_nbr,family,total_unit_sales
0,1503844,produce,6.264200e+06
1,1047679,beverages,5.512875e+06
2,1473474,produce,4.990323e+06
3,364606,grocery i,4.416426e+06
4,819932,beverages,4.410943e+06
5,1463992,beverages,4.408545e+06
6,807493,beverages,4.321424e+06
7,265559,grocery i,4.112827e+06
8,502331,bread/bakery,3.703350e+06
9,314384,grocery i,3.544341e+06


In [36]:
con.execute("""
    CREATE OR REPLACE TABLE bottom_10_items_details AS
    SELECT
        t.item_nbr,
        i.family,
        SUM(t.unit_sales) AS total_unit_sales
    FROM train t
    JOIN items i
        ON t.item_nbr = i.item_nbr
    GROUP BY
        t.item_nbr,
        i.family
    ORDER BY total_unit_sales ASC
    LIMIT 10
""")

In [37]:
con.execute("""
    SELECT *
    FROM bottom_10_items_details
    ORDER BY total_unit_sales ASC
""").fetchdf()

,item_nbr,family,total_unit_sales
0,2015898,grocery i,1.0
1,2011451,grocery i,1.0
2,2123747,grocery i,3.0
3,2122818,beverages,4.0
4,2126944,beverages,5.0
5,2122947,"liquor,wine,beer",5.0
6,2123209,"liquor,wine,beer",6.0
7,2123463,grocery i,8.0
8,2123790,"liquor,wine,beer",9.0
9,2116238,"liquor,wine,beer",10.0


In [38]:
con.execute("""
    CREATE OR REPLACE TABLE promotion_sales AS
    SELECT
        onpromotion,
        COUNT(*) AS row_count,
        SUM(unit_sales) AS total_unit_sales,
        AVG(unit_sales) AS avg_unit_sales_per_record,
        COUNT(DISTINCT date) AS observed_days
    FROM train
    GROUP BY onpromotion
    ORDER BY onpromotion NULLS LAST
""")

In [39]:
con.execute("""
    SELECT *
    FROM promotion_sales
    ORDER BY onpromotion NULLS LAST
""").fetchdf()

,onpromotion,row_count,total_unit_sales,avg_unit_sales_per_record,observed_days
0,False,96028767,7.766937e+08,8.088135,1230
1,True,7810622,1.051830e+08,13.466666,1230
2,<NA>,21657651,1.917336e+08,8.852925,454


In [40]:
con.execute("""
    CREATE OR REPLACE TABLE yearly_product_promotion AS
    SELECT
        YEAR(date) AS year,
        onpromotion,
        COUNT(*) AS row_count,
        SUM(unit_sales) AS total_unit_sales,
        AVG(unit_sales) AS avg_unit_sales_per_record
    FROM train
    GROUP BY
        YEAR(date),
        onpromotion
    ORDER BY
        year,
        onpromotion NULLS LAST
""")

In [41]:
con.execute("""
    SELECT *
    FROM yearly_product_promotion
    ORDER BY year, onpromotion NULLS LAST
""").fetchdf()

,year,onpromotion,row_count,total_unit_sales,avg_unit_sales_per_record
0,2013,<NA>,16322662,1.404180e+08,8.602641
1,2014,False,16477499,1.503467e+08,9.124364
2,2014,True,459114,7.811852e+06,17.015060
3,2014,<NA>,5334989,5.131556e+07,9.618682
4,2015,False,26777369,2.248940e+08,8.398661
5,2015,True,1087275,1.597378e+07,14.691576
6,2016,False,31715287,2.449577e+08,7.723647
7,2016,True,3514584,4.367570e+07,12.426989
8,2017,False,21058612,1.564952e+08,7.431413
9,2017,True,2749649,3.772170e+07,13.718735


In [42]:
con.execute("SHOW TABLES").fetchdf()

,name
0,bottom_10_items_details
1,daily_store_family_sales
2,daily_store_sales
3,family_sales
4,holiday_weekday_effect
5,items
6,monthly_normalized
7,monthly_store_family_sales
8,oil_sales_analysis
9,promotion_sales


In [43]:
from pathlib import Path

project_dir = Path(
    r"D:\Data Analysis Projects_\favorita-grocery-sales-forecasting"
)

[
    str(p) for p in project_dir.rglob("*")
    if p.is_file()
    and ("holiday" in p.name.lower()
         or "event" in p.name.lower())
]

['D:\\Data Analysis Projects_\\favorita-grocery-sales-forecasting\\Notebook\\favorita_exports\\holiday_weekday_effect.csv',
 'D:\\Data Analysis Projects_\\favorita-grocery-sales-forecasting\\Notebook\\favorita_exports\\CSV\\holiday_weekday_effect.csv',
 'D:\\Data Analysis Projects_\\favorita-grocery-sales-forecasting\\Datasets\\holidays_events.csv\\holidays_events.csv']

In [44]:
from pathlib import Path

holiday_effect_path = (
    Path(r"D:\Data Analysis Projects_\favorita-grocery-sales-forecasting")
    / "Notebook"
    / "favorita_exports"
    / "holiday_weekday_effect.csv"
)

print(holiday_effect_path.exists())

True


In [45]:
con.execute(f"""
    CREATE OR REPLACE TABLE holiday_weekday_effect AS
    SELECT *
    FROM read_csv_auto('{holiday_effect_path.as_posix()}')
""")

In [46]:
con.execute("""
    SELECT *
    FROM holiday_weekday_effect
""").fetchdf()

,store_nbr,weekday,avg_daily_sales_false,avg_daily_sales_true,days_count_false,days_count_true,holiday_sales_difference_pct
0,50,3,15046.341603,17522.986429,232,7,16.460113
1,45,3,28919.856099,32073.623400,234,5,10.905197
2,2,1,12384.857792,12397.552300,231,10,0.102500
3,19,4,6384.221009,7241.935667,233,6,13.434915
4,48,2,16037.277931,30904.840500,233,8,92.706272
...,...,...,...,...,...,...,...
266,5,7,11250.509909,11584.366125,231,8,2.967476
267,16,7,9231.422498,9523.850875,231,8,3.167750
268,53,7,12943.908801,12407.958000,161,5,-4.140564
269,15,6,7294.298462,7912.476800,236,5,8.474816


In [47]:
oil_analysis_path = (
    Path(r"D:\Data Analysis Projects_\favorita-grocery-sales-forecasting")
    / "Notebook"
    / "favorita_exports"
    / "oil_sales_analysis.csv"
)

print(oil_analysis_path.exists())

True


In [48]:
con.execute(f"""
    CREATE OR REPLACE TABLE oil_sales_analysis AS
    SELECT *
    FROM read_csv_auto('{oil_analysis_path.as_posix()}')
""")

In [49]:
con.execute("SELECT * FROM oil_sales_analysis LIMIT 10").fetchdf()

,month,total_unit_sales,observed_days,calendar_days,avg_daily_unit_sales,avg_oil_price,observations
0,2013-01-01,1.032741e+07,31,31,333142.314065,94.756667,21
1,2013-02-01,9.658960e+06,28,28,344962.849179,95.308947,19
2,2013-03-01,1.142850e+07,31,31,368661.194742,92.938500,20
3,2013-04-01,1.099346e+07,30,30,366448.757967,92.021364,22
4,2013-05-01,1.159770e+07,31,31,374119.419581,94.509545,22
5,2013-06-01,1.168934e+07,30,30,389644.535400,95.772500,20
6,2013-07-01,1.125740e+07,31,31,363141.890548,104.670909,22
7,2013-08-01,1.173779e+07,31,31,378638.320000,106.572727,22
8,2013-09-01,1.179293e+07,30,30,393097.741100,106.289500,20
9,2013-10-01,1.177562e+07,31,31,379858.689032,100.538261,23


In [50]:
con.execute("SHOW TABLES").fetchdf()

,name
0,bottom_10_items_details
1,daily_store_family_sales
2,daily_store_sales
3,family_sales
4,holiday_weekday_effect
5,items
6,monthly_normalized
7,monthly_store_family_sales
8,oil_sales_analysis
9,promotion_sales


In [51]:
con.execute("""
    SELECT 'train' AS table_name, COUNT(*) AS row_count
    FROM train

    UNION ALL

    SELECT 'stores', COUNT(*)
    FROM stores

    UNION ALL

    SELECT 'items', COUNT(*)
    FROM items

    UNION ALL

    SELECT 'transactions', COUNT(*)
    FROM transactions

    UNION ALL

    SELECT 'daily_store_sales', COUNT(*)
    FROM daily_store_sales
""").fetchdf()

,table_name,row_count
0,train,125497040
1,stores,54
2,items,4100
3,transactions,83488
4,daily_store_sales,83606


In [52]:
con.execute("""
    SELECT 'yearly_sales' AS table_name, COUNT(*) AS row_count FROM yearly_sales
    UNION ALL
    SELECT 'monthly_normalized', COUNT(*) FROM monthly_normalized
    UNION ALL
    SELECT 'weekday_summary', COUNT(*) FROM weekday_summary
    UNION ALL
    SELECT 'store_total_sales', COUNT(*) FROM store_total_sales
    UNION ALL
    SELECT 'store_type_sales', COUNT(*) FROM store_type_sales
    UNION ALL
    SELECT 'store_yearly_sales', COUNT(*) FROM store_yearly_sales
    UNION ALL
    SELECT 'family_sales', COUNT(*) FROM family_sales
    UNION ALL
    SELECT 'top_10_items_details', COUNT(*) FROM top_10_items_details
    UNION ALL
    SELECT 'bottom_10_items_details', COUNT(*) FROM bottom_10_items_details
    UNION ALL
    SELECT 'promotion_sales', COUNT(*) FROM promotion_sales
    UNION ALL
    SELECT 'yearly_product_promotion', COUNT(*) FROM yearly_product_promotion
    UNION ALL
    SELECT 'holiday_weekday_effect', COUNT(*) FROM holiday_weekday_effect
    UNION ALL
    SELECT 'oil_sales_analysis', COUNT(*) FROM oil_sales_analysis
    ORDER BY table_name
""").fetchdf()

,table_name,row_count
0,bottom_10_items_details,10
1,family_sales,33
2,holiday_weekday_effect,271
3,monthly_normalized,56
4,oil_sales_analysis,56
5,promotion_sales,3
6,store_total_sales,54
7,store_type_sales,5
8,store_yearly_sales,255
9,top_10_items_details,10


In [53]:
con.execute("DESCRIBE stores").fetchdf()

,column_name,column_type,null,key,default,extra
0,store_nbr,BIGINT,YES,None,None,None
1,city,VARCHAR,YES,None,None,None
2,state,VARCHAR,YES,None,None,None
3,type,VARCHAR,YES,None,None,None


In [54]:
con.execute("SELECT * FROM stores LIMIT 5").fetchdf()

,store_nbr,city,state,type
0,1,Quito,Pichincha,D
1,2,Quito,Pichincha,D
2,3,Quito,Pichincha,D
3,4,Quito,Pichincha,D
4,5,Santo Domingo,Santo Domingo de los Tsachilas,D


In [55]:
con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT store_nbr) AS unique_stores,
        COUNT(*) - COUNT(DISTINCT store_nbr) AS duplicate_rows,
        COUNT(*) FILTER (WHERE store_nbr IS NULL) AS null_store_ids
    FROM stores
""").fetchdf()

,total_rows,unique_stores,duplicate_rows,null_store_ids
0,54,54,0,0


In [56]:
con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT item_nbr) AS unique_items,
        COUNT(*) - COUNT(DISTINCT item_nbr) AS duplicate_rows,
        COUNT(*) FILTER (WHERE item_nbr IS NULL) AS null_item_ids
    FROM items
""").fetchdf()

,total_rows,unique_items,duplicate_rows,null_item_ids
0,4100,4100,0,0


In [57]:
con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (date, store_nbr)) AS unique_store_days,
        COUNT(*) - COUNT(DISTINCT (date, store_nbr))
            AS duplicate_store_days,
        COUNT(*) FILTER (
            WHERE date IS NULL OR store_nbr IS NULL
        ) AS rows_with_null_keys
    FROM daily_store_sales
""").fetchdf()

,total_rows,unique_store_days,duplicate_store_days,rows_with_null_keys
0,83606,83606,0,0


In [58]:
con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (date, store_nbr)) AS unique_store_days,
        COUNT(*) - COUNT(DISTINCT (date, store_nbr))
            AS duplicate_store_days,
        COUNT(*) FILTER (
            WHERE date IS NULL OR store_nbr IS NULL
        ) AS rows_with_null_keys
    FROM transactions
""").fetchdf()

,total_rows,unique_store_days,duplicate_store_days,rows_with_null_keys
0,83488,83488,0,0


In [59]:
con.execute("""
    WITH sales_keys AS (
        SELECT date, store_nbr
        FROM daily_store_sales
    ),
    transaction_keys AS (
        SELECT date, store_nbr
        FROM transactions
    )
    SELECT
        (
            SELECT COUNT(*)
            FROM transaction_keys t
            ANTI JOIN sales_keys s
            USING (date, store_nbr)
        ) AS transactions_without_sales,
        (
            SELECT COUNT(*)
            FROM sales_keys s
            ANTI JOIN transaction_keys t
            USING (date, store_nbr)
        ) AS sales_without_transactions
""").fetchdf()

,transactions_without_sales,sales_without_transactions
0,0,118


In [60]:
con.execute("""
    SELECT
        s.store_nbr,
        COUNT(*) AS missing_transaction_days,
        MIN(s.date) AS first_date,
        MAX(s.date) AS last_date,
        SUM(s.daily_unit_sales) AS total_unit_sales
    FROM daily_store_sales s
    ANTI JOIN transactions t
        USING (date, store_nbr)
    GROUP BY s.store_nbr
    ORDER BY missing_transaction_days DESC, s.store_nbr
""").fetchdf()

,store_nbr,missing_transaction_days,first_date,last_date,total_unit_sales
0,7,4,2016-01-02,2016-09-27,94328.1400
1,10,4,2013-06-19,2016-01-04,31857.5130
2,3,3,2016-01-02,2016-01-04,151383.6830
3,4,3,2016-01-02,2016-01-04,55246.3920
4,6,3,2016-01-02,2016-01-04,77646.2540
5,8,3,2016-01-02,2016-01-04,86303.7800
6,9,3,2016-01-02,2016-01-04,82496.5360
7,11,3,2016-01-02,2016-01-04,83870.8100
8,12,3,2016-01-02,2016-01-04,29955.0470
9,13,3,2016-01-02,2016-01-04,31303.0080


In [61]:
con.execute("""
    SELECT
        s.date,
        COUNT(*) AS stores_without_transactions,
        SUM(s.daily_unit_sales) AS total_unit_sales
    FROM daily_store_sales s
    ANTI JOIN transactions t
        USING (date, store_nbr)
    GROUP BY s.date
    ORDER BY s.date
""").fetchdf()

,date,stores_without_transactions,total_unit_sales
0,2013-06-19,4,1.312169e+04
1,2014-01-02,1,3.146147e+03
2,2014-03-24,1,8.014160e+03
3,2016-01-01,1,1.643339e+04
4,2016-01-02,17,3.087439e+05
5,2016-01-03,53,1.226736e+06
6,2016-01-04,39,7.784304e+05
7,2016-09-27,2,2.740861e+04


In [62]:
con.execute("""
    SELECT
        date,
        COUNT(*) AS stores_with_transactions,
        COUNT(DISTINCT store_nbr) AS unique_stores
    FROM transactions
    WHERE date BETWEEN DATE '2016-01-01'
                   AND DATE '2016-01-05'
    GROUP BY date
    ORDER BY date
""").fetchdf()

,date,stores_with_transactions,unique_stores
0,2016-01-02,36,36
1,2016-01-04,14,14
2,2016-01-05,53,53


In [63]:
con.execute("""
    WITH dates AS (
        SELECT unnest(
            generate_series(
                DATE '2016-01-01',
                DATE '2016-01-05',
                INTERVAL 1 DAY
            )
        )::DATE AS date
    ),
    sales AS (
        SELECT date, COUNT(DISTINCT store_nbr) AS sales_stores
        FROM daily_store_sales
        WHERE date BETWEEN DATE '2016-01-01' AND DATE '2016-01-05'
        GROUP BY date
    ),
    trans AS (
        SELECT date, COUNT(DISTINCT store_nbr) AS transaction_stores
        FROM transactions
        WHERE date BETWEEN DATE '2016-01-01' AND DATE '2016-01-05'
        GROUP BY date
    )
    SELECT
        d.date,
        COALESCE(s.sales_stores, 0) AS sales_stores,
        COALESCE(t.transaction_stores, 0) AS transaction_stores
    FROM dates d
    LEFT JOIN sales s USING (date)
    LEFT JOIN trans t USING (date)
    ORDER BY d.date
""").fetchdf()

,date,sales_stores,transaction_stores
0,2016-01-01,1,0
1,2016-01-02,53,36
2,2016-01-03,53,0
3,2016-01-04,53,14
4,2016-01-05,53,53


In [64]:
con.execute("DESCRIBE transactions").fetchdf()

,column_name,column_type,null,key,default,extra
0,date,DATE,YES,None,None,None
1,store_nbr,BIGINT,YES,None,None,None
2,transactions,BIGINT,YES,None,None,None


In [65]:
con.execute("""
    SELECT
        MIN(date) AS first_date,
        MAX(date) AS last_date,
        COUNT(*) AS total_rows,
        COUNT(DISTINCT date) AS unique_dates,
        COUNT(DISTINCT store_nbr) AS unique_stores,
        COUNT(*) FILTER (
            WHERE transactions IS NULL
        ) AS null_transaction_values,
        COUNT(*) FILTER (
            WHERE transactions <= 0
        ) AS nonpositive_transaction_values
    FROM transactions
""").fetchdf()

,first_date,last_date,total_rows,unique_dates,unique_stores,null_transaction_values,nonpositive_transaction_values
0,2013-01-01,2017-08-15,83488,1682,54,0,0


In [66]:
con.execute("""
    SELECT
        store_nbr,
        COUNT(*) AS recorded_days,
        MIN(date) AS first_date,
        MAX(date) AS last_date
    FROM transactions
    GROUP BY store_nbr
    ORDER BY recorded_days, store_nbr
""").fetchdf()

,store_nbr,recorded_days,first_date,last_date
0,52,118,2017-04-20,2017-08-15
1,22,671,2015-10-09,2017-08-15
2,42,720,2015-08-21,2017-08-15
3,21,748,2015-07-24,2017-08-15
4,29,874,2015-03-20,2017-08-15
5,20,909,2015-02-13,2017-08-15
6,53,1167,2014-05-29,2017-08-15
7,36,1551,2013-05-09,2017-08-15
8,18,1566,2013-01-02,2017-08-15
9,24,1577,2013-01-02,2017-08-15


In [67]:
con.execute("""
    SELECT
        'daily_store_sales' AS source,
        COUNT(*) AS recorded_days,
        MIN(date) AS first_date,
        MAX(date) AS last_date
    FROM daily_store_sales
    WHERE store_nbr = 52

    UNION ALL

    SELECT
        'transactions' AS source,
        COUNT(*) AS recorded_days,
        MIN(date) AS first_date,
        MAX(date) AS last_date
    FROM transactions
    WHERE store_nbr = 52
""").fetchdf()

,source,recorded_days,first_date,last_date
0,daily_store_sales,118,2017-04-20,2017-08-15
1,transactions,118,2017-04-20,2017-08-15


In [68]:
con.execute("""
    WITH sales AS (
        SELECT
            store_nbr,
            COUNT(*) AS sales_days
        FROM daily_store_sales
        GROUP BY store_nbr
    ),
    trans AS (
        SELECT
            store_nbr,
            COUNT(*) AS transaction_days
        FROM transactions
        GROUP BY store_nbr
    )
    SELECT
        s.store_nbr,
        s.sales_days,
        t.transaction_days,
        s.sales_days - t.transaction_days AS day_difference
    FROM sales s
    JOIN trans t USING (store_nbr)
    WHERE t.transaction_days < 1000
    ORDER BY t.transaction_days
""").fetchdf()

,store_nbr,sales_days,transaction_days,day_difference
0,52,118,118,0
1,22,673,671,2
2,42,722,720,2
3,21,750,748,2
4,29,876,874,2
5,20,911,909,2


In [69]:
import duckdb

print(duckdb.__version__)

2.0.0


In [70]:
import duckdb

parquet_path = r"D:\Data Analysis Projects_\favorita-grocery-sales-forecasting\Notebook\favorita_exports\Parquet\stores_pl.parquet"

con = duckdb.connect(
    r"D:\Data Analysis Projects_\favorita-grocery-sales-forecasting\Notebook\favorita.duckdb"
)

con.execute(
    "SELECT * FROM read_parquet(?) LIMIT 5",
    [parquet_path]
).fetchdf()

,store_nbr,city,state,type
0,1,Quito,Pichincha,D
1,2,Quito,Pichincha,D
2,3,Quito,Pichincha,D
3,4,Quito,Pichincha,D
4,5,Santo Domingo,Santo Domingo de los Tsachilas,D


In [71]:
con.execute("""
    CREATE OR REPLACE TABLE stores AS
    SELECT *
    FROM read_parquet(?)
""", [parquet_path])

In [72]:
con.execute("SELECT * FROM stores LIMIT 5").fetchdf()

,store_nbr,city,state,type
0,1,Quito,Pichincha,D
1,2,Quito,Pichincha,D
2,3,Quito,Pichincha,D
3,4,Quito,Pichincha,D
4,5,Santo Domingo,Santo Domingo de los Tsachilas,D


In [73]:
from pathlib import Path

export_dir = Path(
    r"D:\Data Analysis Projects_\favorita-grocery-sales-forecasting\Notebook\favorita_exports\Parquet"
)

files = {
    "items": "items_pl.parquet",
    "transactions": "transactions_pl.parquet",
    "daily_store_sales": "daily_store_sales.parquet",
}

for table_name, filename in files.items():
    file_path = export_dir / filename

    con.execute(
        f"""
        CREATE OR REPLACE TABLE {table_name} AS
        SELECT *
        FROM read_parquet(?)
        """,
        [str(file_path)]
    )

    print(f"{table_name}: created successfully")

items: created successfully
transactions: created successfully
daily_store_sales: created successfully


In [74]:
tables = [
    "stores",
    "items",
    "transactions",
    "daily_store_sales"
]

for table in tables:
    result = con.execute(f"""
        SELECT
            COUNT(*) AS row_count
        FROM {table}
    """).fetchone()

    columns = con.execute(
        f"DESCRIBE {table}"
    ).fetchdf()

    print(f"\nTable: {table}")
    print(f"Rows: {result[0]}")
    print(f"Columns: {len(columns)}")
    print(columns[["column_name", "column_type"]])


Table: stores
Rows: 54
Columns: 4
  column_name column_type
0   store_nbr      BIGINT
1        city     VARCHAR
2       state     VARCHAR
3        type     VARCHAR

Table: items
Rows: 4100
Columns: 4
  column_name column_type
0    item_nbr      BIGINT
1      family     VARCHAR
2       class      BIGINT
3  perishable      BIGINT

Table: transactions
Rows: 83488
Columns: 3
    column_name column_type
0          date        DATE
1     store_nbr      BIGINT
2  transactions      BIGINT

Table: daily_store_sales
Rows: 83606
Columns: 3
        column_name column_type
0              date        DATE
1         store_nbr      BIGINT
2  daily_unit_sales      DOUBLE


In [75]:
for table in ["stores", "items", "transactions", "daily_store_sales"]:
    row_count = con.execute(
        f"SELECT COUNT(*) FROM {table}"
    ).fetchone()[0]

    print(f"{table}: {row_count:,} rows")

train_path = export_dir / "train_df.parquet"

print(f"\ntrain_df.parquet exists: {train_path.exists()}")
if train_path.exists():
    print(f"File size: {train_path.stat().st_size / (1024**3):.2f} GB")

stores: 54 rows
items: 4,100 rows
transactions: 83,488 rows
daily_store_sales: 83,606 rows

train_df.parquet exists: True
File size: 0.38 GB


In [76]:
train_path = export_dir / "train_df.parquet"

con.execute("""
    SELECT
        COUNT(*) AS row_count,
        MIN(date) AS min_date,
        MAX(date) AS max_date,
        COUNT(DISTINCT store_nbr) AS stores_count,
        COUNT(DISTINCT item_nbr) AS items_count
    FROM read_parquet(?)
""", [str(train_path)]).fetchdf()

,row_count,min_date,max_date,stores_count,items_count
0,125497040,2013-01-01,2017-08-15,54,4036


In [77]:
con.execute("""
    SELECT
        SUM(
            COALESCE(LENGTH(CAST(id AS VARCHAR)), 0) +
            COALESCE(LENGTH(CAST(date AS VARCHAR)), 0) +
            COALESCE(LENGTH(CAST(store_nbr AS VARCHAR)), 0) +
            COALESCE(LENGTH(CAST(item_nbr AS VARCHAR)), 0) +
            COALESCE(LENGTH(CAST(unit_sales AS VARCHAR)), 0) +
            COALESCE(LENGTH(CAST(onpromotion AS VARCHAR)), 0)
        ) AS estimated_text_bytes
    FROM read_parquet(?)
""", [str(train_path)]).fetchdf()

,estimated_text_bytes
0,4.244470e+09


In [78]:
con.execute(f"""
    CREATE OR REPLACE VIEW train AS
    SELECT *
    FROM read_parquet('{train_path.as_posix()}')
""")

In [79]:
con.execute("""
    SELECT
        COUNT(*) AS row_count,
        SUM(unit_sales) AS total_unit_sales
    FROM train
""").fetchdf()

,row_count,total_unit_sales
0,125497040,1.073610e+09


In [80]:
con.close()

con = duckdb.connect(
    r"D:\Data Analysis Projects_\favorita-grocery-sales-forecasting\Notebook\favorita.duckdb"
)

In [81]:
con.execute("""
    SELECT COUNT(*) AS row_count
    FROM train
""").fetchdf()

,row_count
0,125497040


In [82]:
con.execute("SHOW TABLES").fetchdf()

,name
0,bottom_10_items_details
1,daily_store_family_sales
2,daily_store_sales
3,family_sales
4,holiday_weekday_effect
5,items
6,monthly_normalized
7,monthly_store_family_sales
8,oil_sales_analysis
9,promotion_sales


In [83]:
con.execute("""
    SELECT
        YEAR(date) AS year,
        SUM(unit_sales) AS total_unit_sales
    FROM train
    GROUP BY YEAR(date)
    ORDER BY year
""").fetchdf()

,year,total_unit_sales
0,2013,1.404180e+08
1,2014,2.094741e+08
2,2015,2.408678e+08
3,2016,2.886334e+08
4,2017,1.942169e+08


In [84]:
con.execute("""
    SELECT
        YEAR(date) AS year,
        SUM(unit_sales) AS total_unit_sales
    FROM train
    WHERE date >= DATE '2016-01-01'
      AND date <= DATE '2017-08-15'
      AND (
          (YEAR(date) = 2016 AND date <= DATE '2016-08-15')
          OR YEAR(date) = 2017
      )
    GROUP BY YEAR(date)
    ORDER BY year
""").fetchdf()

,year,total_unit_sales
0,2016,1.765439e+08
1,2017,1.942169e+08


In [85]:
con.execute("""
    CREATE OR REPLACE TABLE yearly_sales AS
    SELECT
        YEAR(date) AS year,
        SUM(unit_sales) AS total_unit_sales
    FROM train
    GROUP BY YEAR(date)
    ORDER BY year
""")

In [86]:
con.execute("""
    SELECT *
    FROM yearly_sales
    ORDER BY year
""").fetchdf()

,year,total_unit_sales
0,2013,1.404180e+08
1,2014,2.094741e+08
2,2015,2.408678e+08
3,2016,2.886334e+08
4,2017,1.942169e+08


In [87]:
con.execute("""
    SELECT
        YEAR(date) AS year,
        MONTH(date) AS month,
        SUM(unit_sales) AS total_unit_sales,
        COUNT(DISTINCT date) AS observed_days,
        DAY(LAST_DAY(MIN(date))) AS calendar_days,
        SUM(unit_sales) / COUNT(DISTINCT date)
            AS avg_per_observed_day,
        SUM(unit_sales) / DAY(LAST_DAY(MIN(date)))
            AS avg_per_calendar_day
    FROM train
    GROUP BY
        YEAR(date),
        MONTH(date)
    ORDER BY
        year,
        month
""").fetchdf()

,year,month,total_unit_sales,observed_days,calendar_days,avg_per_observed_day,avg_per_calendar_day
0,2013,1,1.032741e+07,31,31,333142.314065,333142.314065
1,2013,2,9.658960e+06,28,28,344962.849179,344962.849179
2,2013,3,1.142850e+07,31,31,368661.194742,368661.194742
3,2013,4,1.099346e+07,30,30,366448.757967,366448.757967
4,2013,5,1.159770e+07,31,31,374119.419581,374119.419581
5,2013,6,1.168934e+07,30,30,389644.535400,389644.535400
6,2013,7,1.125740e+07,31,31,363141.890548,363141.890548
7,2013,8,1.173779e+07,31,31,378638.320000,378638.320000
8,2013,9,1.179293e+07,30,30,393097.741100,393097.741100
9,2013,10,1.177562e+07,31,31,379858.689032,379858.689032


In [88]:
con.execute("""
    CREATE OR REPLACE TABLE monthly_normalized AS
    SELECT
        YEAR(date) AS year,
        MONTH(date) AS month,
        SUM(unit_sales) AS total_unit_sales,
        COUNT(DISTINCT date) AS observed_days,
        SUM(unit_sales) / COUNT(DISTINCT date)
            AS avg_daily_unit_sales
    FROM train
    GROUP BY
        YEAR(date),
        MONTH(date)
    ORDER BY
        year,
        month
""")

In [89]:
con.execute("""
    (SELECT * FROM monthly_normalized ORDER BY year, month LIMIT 5)
    UNION ALL
    (SELECT * FROM monthly_normalized ORDER BY year DESC, month DESC LIMIT 5)
""").fetchdf()

,year,month,total_unit_sales,observed_days,avg_daily_unit_sales
0,2013,1,1.032741e+07,31,333142.314065
1,2013,2,9.658960e+06,28,344962.849179
2,2013,3,1.142850e+07,31,368661.194742
3,2013,4,1.099346e+07,30,366448.757967
4,2013,5,1.159770e+07,31,374119.419581
5,2017,8,1.243332e+07,15,828888.195933
6,2017,7,2.701147e+07,31,871337.889226
7,2017,6,2.568277e+07,30,856092.270867
8,2017,5,2.691184e+07,31,868123.846774
9,2017,4,2.589531e+07,30,863176.849700


In [90]:
con.execute("""
    CREATE OR REPLACE TABLE weekday_summary AS
    SELECT
        DAYNAME(date) AS day_name,
        DAYOFWEEK(date) AS day_number,
        SUM(unit_sales) AS total_unit_sales,
        COUNT(DISTINCT date) AS observed_days,
        SUM(unit_sales) / COUNT(DISTINCT date)
            AS avg_daily_unit_sales
    FROM train
    GROUP BY
        DAYNAME(date),
        DAYOFWEEK(date)
    ORDER BY
        day_number
""")

In [91]:
con.execute("""
    SELECT *
    FROM weekday_summary
    ORDER BY day_number
""").fetchdf()

,day_name,day_number,total_unit_sales,observed_days,avg_daily_unit_sales
0,Sunday,0,1.980523e+08,240,825218.075771
1,Monday,1,1.488127e+08,241,617479.929934
2,Tuesday,2,1.379198e+08,242,569916.489993
3,Wednesday,3,1.423786e+08,240,593244.190965
4,Thursday,4,1.212616e+08,240,505256.688612
5,Friday,5,1.390978e+08,240,579573.986125
6,Saturday,6,1.860875e+08,241,772147.340838


In [92]:
con.execute("""
    CREATE OR REPLACE TABLE store_total_sales AS
    WITH store_sales AS (
        SELECT
            store_nbr,
            SUM(unit_sales) AS total_unit_sales
        FROM train
        GROUP BY store_nbr
    )
    SELECT
        store_nbr,
        total_unit_sales,
        100.0 * total_unit_sales
            / SUM(total_unit_sales) OVER ()
            AS sales_contribution_pct,
        RANK() OVER (
            ORDER BY total_unit_sales DESC
        ) AS sales_rank
    FROM store_sales
    ORDER BY sales_rank
""")

In [93]:
con.execute("""
    SELECT *
    FROM store_total_sales
    ORDER BY sales_rank
    LIMIT 10
""").fetchdf()

,store_nbr,total_unit_sales,sales_contribution_pct,sales_rank
0,44,6.208754e+07,5.783062,1
1,45,5.449801e+07,5.076145,2
2,47,5.094831e+07,4.745512,3
3,3,5.048190e+07,4.702069,4
4,49,4.342009e+07,4.044306,5
5,46,4.189605e+07,3.902352,6
6,48,3.593313e+07,3.346944,7
7,51,3.291148e+07,3.065496,8
8,8,3.049134e+07,2.840075,9
9,50,2.865302e+07,2.668847,10


In [94]:
con.execute("""
    SELECT
        COUNT(*) AS store_count,
        SUM(total_unit_sales) AS total_unit_sales,
        SUM(sales_contribution_pct) AS total_contribution_pct,
        SUM(
            CASE WHEN sales_rank <= 10
                 THEN sales_contribution_pct
                 ELSE 0
            END
        ) AS top_10_contribution_pct
    FROM store_total_sales
""").fetchdf()

,store_count,total_unit_sales,total_contribution_pct,top_10_contribution_pct
0,54,1.073610e+09,100.0,40.174809


In [95]:
con.execute("""
    CREATE OR REPLACE TABLE store_type_sales AS
    SELECT
        s.type AS store_type,
        COUNT(*) AS store_count,
        SUM(st.total_unit_sales) AS total_unit_sales,
        SUM(st.total_unit_sales) / COUNT(*) AS avg_sales_per_store,
        100.0 * SUM(st.total_unit_sales)
            / SUM(SUM(st.total_unit_sales)) OVER ()
            AS sales_contribution_pct
    FROM store_total_sales st
    JOIN stores s
        ON st.store_nbr = s.store_nbr
    GROUP BY s.type
    ORDER BY total_unit_sales DESC
""")

In [96]:
con.execute("""
    SELECT *
    FROM store_type_sales
    ORDER BY total_unit_sales DESC
""").fetchdf()

,store_type,store_count,total_unit_sales,avg_sales_per_store,sales_contribution_pct
0,A,9,3.530438e+08,3.922709e+07,32.883796
1,D,18,3.510633e+08,1.950352e+07,32.699326
2,C,15,1.644345e+08,1.096230e+07,15.316028
3,B,8,1.452483e+08,1.815603e+07,13.528956
4,E,4,5.982042e+07,1.495511e+07,5.571894


In [97]:
con.execute("""
    CREATE OR REPLACE TABLE store_yearly_sales AS
    SELECT
        store_nbr,
        YEAR(date) AS year,
        SUM(unit_sales) AS total_unit_sales
    FROM train
    GROUP BY
        store_nbr,
        YEAR(date)
    ORDER BY
        store_nbr,
        year
""")

In [98]:
con.execute("""
    SELECT *
    FROM store_yearly_sales
    ORDER BY store_nbr, year
    LIMIT 10
""").fetchdf()

,store_nbr,year,total_unit_sales
0,1,2013,1.894713e+06
1,1,2014,2.899710e+06
2,1,2015,3.120580e+06
3,1,2016,3.793493e+06
4,1,2017,2.436517e+06
5,2,2013,2.865393e+06
6,2,2014,4.426115e+06
7,2,2015,4.815042e+06
8,2,2016,5.883244e+06
9,2,2017,3.564823e+06


In [99]:
con.execute("""
    CREATE OR REPLACE TABLE family_sales AS
    WITH family_totals AS (
        SELECT
            i.family,
            SUM(t.unit_sales) AS total_unit_sales,
            COUNT(DISTINCT t.item_nbr) AS number_of_products
        FROM train t
        JOIN items i
            ON t.item_nbr = i.item_nbr
        GROUP BY i.family
    ),
    family_contributions AS (
        SELECT
            family,
            total_unit_sales,
            100.0 * total_unit_sales
                / SUM(total_unit_sales) OVER ()
                AS sales_contribution_pct,
            number_of_products,
            total_unit_sales / number_of_products
                AS avg_units_per_product
        FROM family_totals
    )
    SELECT
        *,
        SUM(sales_contribution_pct) OVER (
            ORDER BY total_unit_sales DESC
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS cumulative_contribution_pct
    FROM family_contributions
    ORDER BY total_unit_sales DESC
""")

In [100]:
con.execute("""
    SELECT *
    FROM family_sales
    ORDER BY total_unit_sales DESC
""").fetchdf()

,family,total_unit_sales,sales_contribution_pct,number_of_products,avg_units_per_product,cumulative_contribution_pct
0,grocery i,3.434617e+08,31.991286,1290,266249.401462,31.991286
1,beverages,2.169422e+08,20.206797,604,359175.903974,52.198083
2,produce,1.227047e+08,11.429164,306,400995.681955,63.627247
3,cleaning,9.751840e+07,9.083221,446,218651.116592,72.710468
4,dairy,6.448726e+07,6.006580,242,266476.264463,78.717048
5,bread/bakery,4.213395e+07,3.924510,134,314432.429657,82.641558
6,poultry,3.187600e+07,2.969048,54,590296.379102,85.610606
7,meats,3.108647e+07,2.895508,84,370077.004854,88.506114
8,personal care,2.458094e+07,2.289559,153,160659.758170,90.795673
9,deli,2.411032e+07,2.245724,91,264948.598558,93.041397


In [101]:
result = con.execute("""
    SELECT
        COUNT(*) AS row_count,
        SUM(daily_unit_sales) AS total_units_sold
    FROM daily_store_sales
    WHERE date >= DATE '2016-01-01'
      AND date <= DATE '2016-08-15'
""").fetchdf()

print(result)

   row_count  total_units_sold
0      12029      1.765439e+08


In [102]:
con.execute("""
    CREATE OR REPLACE TABLE top_10_items_details AS
    SELECT
        t.item_nbr,
        i.family,
        SUM(t.unit_sales) AS total_unit_sales
    FROM train t
    JOIN items i
        ON t.item_nbr = i.item_nbr
    GROUP BY
        t.item_nbr,
        i.family
    ORDER BY total_unit_sales DESC
    LIMIT 10
""")

In [103]:
con.execute("""
    SELECT *
    FROM top_10_items_details
    ORDER BY total_unit_sales DESC
""").fetchdf()

,item_nbr,family,total_unit_sales
0,1503844,produce,6.264200e+06
1,1047679,beverages,5.512875e+06
2,1473474,produce,4.990323e+06
3,364606,grocery i,4.416426e+06
4,819932,beverages,4.410943e+06
5,1463992,beverages,4.408545e+06
6,807493,beverages,4.321424e+06
7,265559,grocery i,4.112827e+06
8,502331,bread/bakery,3.703350e+06
9,314384,grocery i,3.544341e+06


In [104]:
con.execute("""
    CREATE OR REPLACE TABLE bottom_10_items_details AS
    SELECT
        t.item_nbr,
        i.family,
        SUM(t.unit_sales) AS total_unit_sales
    FROM train t
    JOIN items i
        ON t.item_nbr = i.item_nbr
    GROUP BY
        t.item_nbr,
        i.family
    ORDER BY total_unit_sales ASC
    LIMIT 10
""")

In [105]:
con.execute("""
    SELECT *
    FROM bottom_10_items_details
    ORDER BY total_unit_sales ASC
""").fetchdf()

,item_nbr,family,total_unit_sales
0,2011451,grocery i,1.0
1,2015898,grocery i,1.0
2,2123747,grocery i,3.0
3,2122818,beverages,4.0
4,2126944,beverages,5.0
5,2122947,"liquor,wine,beer",5.0
6,2123209,"liquor,wine,beer",6.0
7,2123463,grocery i,8.0
8,2123790,"liquor,wine,beer",9.0
9,2116238,"liquor,wine,beer",10.0


In [106]:
con.execute("""
    CREATE OR REPLACE TABLE promotion_sales AS
    SELECT
        onpromotion,
        COUNT(*) AS row_count,
        SUM(unit_sales) AS total_unit_sales,
        AVG(unit_sales) AS avg_unit_sales_per_record,
        COUNT(DISTINCT date) AS observed_days
    FROM train
    GROUP BY onpromotion
    ORDER BY onpromotion NULLS LAST
""")

In [107]:
con.execute("""
    SELECT *
    FROM promotion_sales
    ORDER BY onpromotion NULLS LAST
""").fetchdf()

,onpromotion,row_count,total_unit_sales,avg_unit_sales_per_record,observed_days
0,False,96028767,7.766937e+08,8.088135,1230
1,True,7810622,1.051830e+08,13.466666,1230
2,<NA>,21657651,1.917336e+08,8.852925,454


In [108]:
con.execute("""
    CREATE OR REPLACE TABLE yearly_product_promotion AS
    SELECT
        YEAR(date) AS year,
        onpromotion,
        COUNT(*) AS row_count,
        SUM(unit_sales) AS total_unit_sales,
        AVG(unit_sales) AS avg_unit_sales_per_record
    FROM train
    GROUP BY
        YEAR(date),
        onpromotion
    ORDER BY
        year,
        onpromotion NULLS LAST
""")

In [109]:
con.execute("""
    SELECT *
    FROM yearly_product_promotion
    ORDER BY year, onpromotion NULLS LAST
""").fetchdf()

,year,onpromotion,row_count,total_unit_sales,avg_unit_sales_per_record
0,2013,<NA>,16322662,1.404180e+08,8.602641
1,2014,False,16477499,1.503467e+08,9.124364
2,2014,True,459114,7.811852e+06,17.015060
3,2014,<NA>,5334989,5.131556e+07,9.618682
4,2015,False,26777369,2.248940e+08,8.398661
5,2015,True,1087275,1.597378e+07,14.691576
6,2016,False,31715287,2.449577e+08,7.723647
7,2016,True,3514584,4.367570e+07,12.426989
8,2017,False,21058612,1.564952e+08,7.431413
9,2017,True,2749649,3.772170e+07,13.718735


In [110]:
con.execute("SHOW TABLES").fetchdf()

,name
0,bottom_10_items_details
1,daily_store_family_sales
2,daily_store_sales
3,family_sales
4,holiday_weekday_effect
5,items
6,monthly_normalized
7,monthly_store_family_sales
8,oil_sales_analysis
9,promotion_sales


In [111]:
from pathlib import Path

project_dir = Path(
    r"D:\Data Analysis Projects_\favorita-grocery-sales-forecasting"
)

[
    str(p) for p in project_dir.rglob("*")
    if p.is_file()
    and ("holiday" in p.name.lower()
         or "event" in p.name.lower())
]

['D:\\Data Analysis Projects_\\favorita-grocery-sales-forecasting\\Notebook\\favorita_exports\\holiday_weekday_effect.csv',
 'D:\\Data Analysis Projects_\\favorita-grocery-sales-forecasting\\Notebook\\favorita_exports\\CSV\\holiday_weekday_effect.csv',
 'D:\\Data Analysis Projects_\\favorita-grocery-sales-forecasting\\Datasets\\holidays_events.csv\\holidays_events.csv']

In [112]:
from pathlib import Path

holiday_effect_path = (
    Path(r"D:\Data Analysis Projects_\favorita-grocery-sales-forecasting")
    / "Notebook"
    / "favorita_exports"
    / "holiday_weekday_effect.csv"
)

print(holiday_effect_path.exists())

True


In [113]:
con.execute(f"""
    CREATE OR REPLACE TABLE holiday_weekday_effect AS
    SELECT *
    FROM read_csv_auto('{holiday_effect_path.as_posix()}')
""")

In [114]:
con.execute("""
    SELECT *
    FROM holiday_weekday_effect
""").fetchdf()

,store_nbr,weekday,avg_daily_sales_false,avg_daily_sales_true,days_count_false,days_count_true,holiday_sales_difference_pct
0,50,3,15046.341603,17522.986429,232,7,16.460113
1,45,3,28919.856099,32073.623400,234,5,10.905197
2,2,1,12384.857792,12397.552300,231,10,0.102500
3,19,4,6384.221009,7241.935667,233,6,13.434915
4,48,2,16037.277931,30904.840500,233,8,92.706272
...,...,...,...,...,...,...,...
266,5,7,11250.509909,11584.366125,231,8,2.967476
267,16,7,9231.422498,9523.850875,231,8,3.167750
268,53,7,12943.908801,12407.958000,161,5,-4.140564
269,15,6,7294.298462,7912.476800,236,5,8.474816


In [115]:
oil_analysis_path = (
    Path(r"D:\Data Analysis Projects_\favorita-grocery-sales-forecasting")
    / "Notebook"
    / "favorita_exports"
    / "oil_sales_analysis.csv"
)

print(oil_analysis_path.exists())

True


In [116]:
con.execute(f"""
    CREATE OR REPLACE TABLE oil_sales_analysis AS
    SELECT *
    FROM read_csv_auto('{oil_analysis_path.as_posix()}')
""")

In [117]:
con.execute("SELECT * FROM oil_sales_analysis LIMIT 10").fetchdf()

,month,total_unit_sales,observed_days,calendar_days,avg_daily_unit_sales,avg_oil_price,observations
0,2013-01-01,1.032741e+07,31,31,333142.314065,94.756667,21
1,2013-02-01,9.658960e+06,28,28,344962.849179,95.308947,19
2,2013-03-01,1.142850e+07,31,31,368661.194742,92.938500,20
3,2013-04-01,1.099346e+07,30,30,366448.757967,92.021364,22
4,2013-05-01,1.159770e+07,31,31,374119.419581,94.509545,22
5,2013-06-01,1.168934e+07,30,30,389644.535400,95.772500,20
6,2013-07-01,1.125740e+07,31,31,363141.890548,104.670909,22
7,2013-08-01,1.173779e+07,31,31,378638.320000,106.572727,22
8,2013-09-01,1.179293e+07,30,30,393097.741100,106.289500,20
9,2013-10-01,1.177562e+07,31,31,379858.689032,100.538261,23


In [118]:
con.execute("SHOW TABLES").fetchdf()

,name
0,bottom_10_items_details
1,daily_store_family_sales
2,daily_store_sales
3,family_sales
4,holiday_weekday_effect
5,items
6,monthly_normalized
7,monthly_store_family_sales
8,oil_sales_analysis
9,promotion_sales


In [119]:
con.execute("""
    SELECT 'train' AS table_name, COUNT(*) AS row_count
    FROM train

    UNION ALL

    SELECT 'stores', COUNT(*)
    FROM stores

    UNION ALL

    SELECT 'items', COUNT(*)
    FROM items

    UNION ALL

    SELECT 'transactions', COUNT(*)
    FROM transactions

    UNION ALL

    SELECT 'daily_store_sales', COUNT(*)
    FROM daily_store_sales
""").fetchdf()

,table_name,row_count
0,train,125497040
1,stores,54
2,items,4100
3,transactions,83488
4,daily_store_sales,83606


In [120]:
con.execute("""
    SELECT 'yearly_sales' AS table_name, COUNT(*) AS row_count FROM yearly_sales
    UNION ALL
    SELECT 'monthly_normalized', COUNT(*) FROM monthly_normalized
    UNION ALL
    SELECT 'weekday_summary', COUNT(*) FROM weekday_summary
    UNION ALL
    SELECT 'store_total_sales', COUNT(*) FROM store_total_sales
    UNION ALL
    SELECT 'store_type_sales', COUNT(*) FROM store_type_sales
    UNION ALL
    SELECT 'store_yearly_sales', COUNT(*) FROM store_yearly_sales
    UNION ALL
    SELECT 'family_sales', COUNT(*) FROM family_sales
    UNION ALL
    SELECT 'top_10_items_details', COUNT(*) FROM top_10_items_details
    UNION ALL
    SELECT 'bottom_10_items_details', COUNT(*) FROM bottom_10_items_details
    UNION ALL
    SELECT 'promotion_sales', COUNT(*) FROM promotion_sales
    UNION ALL
    SELECT 'yearly_product_promotion', COUNT(*) FROM yearly_product_promotion
    UNION ALL
    SELECT 'holiday_weekday_effect', COUNT(*) FROM holiday_weekday_effect
    UNION ALL
    SELECT 'oil_sales_analysis', COUNT(*) FROM oil_sales_analysis
    ORDER BY table_name
""").fetchdf()

,table_name,row_count
0,bottom_10_items_details,10
1,family_sales,33
2,holiday_weekday_effect,271
3,monthly_normalized,56
4,oil_sales_analysis,56
5,promotion_sales,3
6,store_total_sales,54
7,store_type_sales,5
8,store_yearly_sales,255
9,top_10_items_details,10


In [121]:
con.execute("DESCRIBE stores").fetchdf()

,column_name,column_type,null,key,default,extra
0,store_nbr,BIGINT,YES,None,None,None
1,city,VARCHAR,YES,None,None,None
2,state,VARCHAR,YES,None,None,None
3,type,VARCHAR,YES,None,None,None


In [122]:
con.execute("SELECT * FROM stores LIMIT 5").fetchdf()

,store_nbr,city,state,type
0,1,Quito,Pichincha,D
1,2,Quito,Pichincha,D
2,3,Quito,Pichincha,D
3,4,Quito,Pichincha,D
4,5,Santo Domingo,Santo Domingo de los Tsachilas,D


In [123]:
con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT store_nbr) AS unique_stores,
        COUNT(*) - COUNT(DISTINCT store_nbr) AS duplicate_rows,
        COUNT(*) FILTER (WHERE store_nbr IS NULL) AS null_store_ids
    FROM stores
""").fetchdf()

,total_rows,unique_stores,duplicate_rows,null_store_ids
0,54,54,0,0


In [124]:
con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT item_nbr) AS unique_items,
        COUNT(*) - COUNT(DISTINCT item_nbr) AS duplicate_rows,
        COUNT(*) FILTER (WHERE item_nbr IS NULL) AS null_item_ids
    FROM items
""").fetchdf()

,total_rows,unique_items,duplicate_rows,null_item_ids
0,4100,4100,0,0


In [125]:
con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (date, store_nbr)) AS unique_store_days,
        COUNT(*) - COUNT(DISTINCT (date, store_nbr))
            AS duplicate_store_days,
        COUNT(*) FILTER (
            WHERE date IS NULL OR store_nbr IS NULL
        ) AS rows_with_null_keys
    FROM daily_store_sales
""").fetchdf()

,total_rows,unique_store_days,duplicate_store_days,rows_with_null_keys
0,83606,83606,0,0


In [126]:
con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (date, store_nbr)) AS unique_store_days,
        COUNT(*) - COUNT(DISTINCT (date, store_nbr))
            AS duplicate_store_days,
        COUNT(*) FILTER (
            WHERE date IS NULL OR store_nbr IS NULL
        ) AS rows_with_null_keys
    FROM transactions
""").fetchdf()

,total_rows,unique_store_days,duplicate_store_days,rows_with_null_keys
0,83488,83488,0,0


In [127]:
con.execute("""
    WITH sales_keys AS (
        SELECT date, store_nbr
        FROM daily_store_sales
    ),
    transaction_keys AS (
        SELECT date, store_nbr
        FROM transactions
    )
    SELECT
        (
            SELECT COUNT(*)
            FROM transaction_keys t
            ANTI JOIN sales_keys s
            USING (date, store_nbr)
        ) AS transactions_without_sales,
        (
            SELECT COUNT(*)
            FROM sales_keys s
            ANTI JOIN transaction_keys t
            USING (date, store_nbr)
        ) AS sales_without_transactions
""").fetchdf()

,transactions_without_sales,sales_without_transactions
0,0,118


In [128]:
con.execute("""
    SELECT
        s.store_nbr,
        COUNT(*) AS missing_transaction_days,
        MIN(s.date) AS first_date,
        MAX(s.date) AS last_date,
        SUM(s.daily_unit_sales) AS total_unit_sales
    FROM daily_store_sales s
    ANTI JOIN transactions t
        USING (date, store_nbr)
    GROUP BY s.store_nbr
    ORDER BY missing_transaction_days DESC, s.store_nbr
""").fetchdf()

,store_nbr,missing_transaction_days,first_date,last_date,total_unit_sales
0,7,4,2016-01-02,2016-09-27,94328.1400
1,10,4,2013-06-19,2016-01-04,31857.5130
2,3,3,2016-01-02,2016-01-04,151383.6830
3,4,3,2016-01-02,2016-01-04,55246.3920
4,6,3,2016-01-02,2016-01-04,77646.2540
5,8,3,2016-01-02,2016-01-04,86303.7800
6,9,3,2016-01-02,2016-01-04,82496.5360
7,11,3,2016-01-02,2016-01-04,83870.8100
8,12,3,2016-01-02,2016-01-04,29955.0470
9,13,3,2016-01-02,2016-01-04,31303.0080


In [129]:
con.execute("""
    SELECT
        s.date,
        COUNT(*) AS stores_without_transactions,
        SUM(s.daily_unit_sales) AS total_unit_sales
    FROM daily_store_sales s
    ANTI JOIN transactions t
        USING (date, store_nbr)
    GROUP BY s.date
    ORDER BY s.date
""").fetchdf()

,date,stores_without_transactions,total_unit_sales
0,2013-06-19,4,1.312169e+04
1,2014-01-02,1,3.146147e+03
2,2014-03-24,1,8.014160e+03
3,2016-01-01,1,1.643339e+04
4,2016-01-02,17,3.087439e+05
5,2016-01-03,53,1.226736e+06
6,2016-01-04,39,7.784304e+05
7,2016-09-27,2,2.740861e+04


In [130]:
con.execute("""
    SELECT
        s.store_nbr,
        s.date,
        s.daily_unit_sales
    FROM daily_store_sales s
    ANTI JOIN transactions t
        USING (date, store_nbr)
    WHERE s.store_nbr IN (20, 21, 22, 29, 42)
    ORDER BY s.store_nbr, s.date
""").fetchdf()

,store_nbr,date,daily_unit_sales
0,20,2016-01-03,29730.362
1,20,2016-01-04,15215.205
2,21,2016-01-03,21699.477
3,21,2016-01-04,13750.333
4,22,2016-01-03,8077.005
5,22,2016-01-04,6361.602
6,29,2016-01-03,15961.380
7,29,2016-01-04,10977.714
8,42,2016-01-03,18376.455
9,42,2016-01-04,15302.489


In [131]:
con.execute("""
    SELECT sql
    FROM duckdb_tables()
    WHERE table_name = 'transactions'
""").fetchdf()

,sql
0,"CREATE TABLE transactions(date DATE, store_nbr..."


In [132]:
con.execute("""
    SELECT
        date,
        store_nbr,
        transactions
    FROM transactions
    WHERE date BETWEEN DATE '2016-01-01'
                   AND DATE '2016-01-05'
    ORDER BY date, store_nbr
""").fetchdf()

,date,store_nbr,transactions
0,2016-01-02,2,6
1,2016-01-02,16,373
2,2016-01-02,20,1816
3,2016-01-02,21,1388
4,2016-01-02,22,849
...,...,...,...
98,2016-01-05,49,2967
99,2016-01-05,50,2528
100,2016-01-05,51,1887
101,2016-01-05,53,875


In [133]:
transactions_path = export_dir / "transactions_pl.parquet"

print(
    con.execute(
        """
        SELECT
            COUNT(*) AS rows,
            COUNT(DISTINCT (date, store_nbr)) AS unique_keys,
            MIN(date) AS min_date,
            MAX(date) AS max_date
        FROM read_parquet(?)
        """,
        [str(transactions_path)]
    ).df()
)

    rows  unique_keys   min_date   max_date
0  83488        83488 2013-01-01 2017-08-15


In [134]:
con.execute("DESCRIBE train").df()

,column_name,column_type,null,key,default,extra
0,id,BIGINT,YES,None,None,None
1,date,DATE,YES,None,None,None
2,store_nbr,BIGINT,YES,None,None,None
3,item_nbr,BIGINT,YES,None,None,None
4,unit_sales,DOUBLE,YES,None,None,None
5,onpromotion,BOOLEAN,YES,None,None,None
6,year,INTEGER,YES,None,None,None
7,month,TINYINT,YES,None,None,None
8,month_name,VARCHAR,YES,None,None,None
9,day,TINYINT,YES,None,None,None


In [135]:
con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT id) AS unique_ids,
        COUNT(*) FILTER (
            WHERE date IS NULL
               OR store_nbr IS NULL
               OR item_nbr IS NULL
        ) AS rows_missing_keys
    FROM train
""").df()

,total_rows,unique_ids,rows_missing_keys
0,125497040,125497040,0


In [136]:
con.execute("""
    SELECT
        table_name,
        table_type
    FROM information_schema.tables
    WHERE table_schema = 'main'
    ORDER BY table_type, table_name
""").df()

,table_name,table_type
0,bottom_10_items_details,BASE TABLE
1,daily_store_family_sales,BASE TABLE
2,daily_store_sales,BASE TABLE
3,family_sales,BASE TABLE
4,holiday_weekday_effect,BASE TABLE
5,items,BASE TABLE
6,monthly_normalized,BASE TABLE
7,monthly_store_family_sales,BASE TABLE
8,oil_sales_analysis,BASE TABLE
9,promotion_sales,BASE TABLE


In [137]:
tables = con.execute("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'main'
      AND table_name <> 'train'
    ORDER BY table_name
""").fetchall()

for (table_name,) in tables:
    result = con.execute(f"""
        SELECT COUNT(*) AS row_count
        FROM "{table_name}"
    """).fetchone()[0]

    columns = con.execute("""
        SELECT column_name
        FROM information_schema.columns
        WHERE table_schema = 'main'
          AND table_name = ?
        ORDER BY ordinal_position
    """, [table_name]).fetchall()

    print(f"{table_name} | rows={result} | columns={[c[0] for c in columns]}")

bottom_10_items_details | rows=10 | columns=['item_nbr', 'family', 'total_unit_sales']
daily_store_family_sales | rows=2061903 | columns=['date', 'store_nbr', 'family', 'daily_unit_sales', 'product_records', 'distinct_items']
daily_store_sales | rows=83606 | columns=['date', 'store_nbr', 'daily_unit_sales']
family_sales | rows=33 | columns=['family', 'total_unit_sales', 'sales_contribution_pct', 'number_of_products', 'avg_units_per_product', 'cumulative_contribution_pct']
holiday_weekday_effect | rows=271 | columns=['store_nbr', 'weekday', 'avg_daily_sales_false', 'avg_daily_sales_true', 'days_count_false', 'days_count_true', 'holiday_sales_difference_pct']
items | rows=4100 | columns=['item_nbr', 'family', 'class', 'perishable']
monthly_normalized | rows=56 | columns=['year', 'month', 'total_unit_sales', 'observed_days', 'avg_daily_unit_sales']
monthly_store_family_sales | rows=76133 | columns=['year', 'month', 'store_nbr', 'family', 'total_unit_sales', 'observed_days', 'observed_day_

In [138]:
con.execute("""
    SELECT
        COUNT(*) AS sales_rows,
        COUNT(DISTINCT date || '-' || store_nbr) AS sales_store_days
    FROM daily_store_sales
""").df()

,sales_rows,sales_store_days
0,83606,83606


In [139]:
con.execute("""
    SELECT
        COUNT(*) AS transaction_rows,
        COUNT(DISTINCT (date, store_nbr)) AS transaction_store_days
    FROM transactions
""").df()

,transaction_rows,transaction_store_days
0,83488,83488


In [140]:
con.execute("""
    SELECT
        COUNT(*) AS sales_rows,
        COUNT(t.transactions) AS matched_rows,
        COUNT(*) - COUNT(t.transactions) AS unmatched_rows
    FROM daily_store_sales s
    LEFT JOIN transactions t
        ON s.date = t.date
       AND s.store_nbr = t.store_nbr
""").df()

,sales_rows,matched_rows,unmatched_rows
0,83606,83488,118


In [141]:
con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT date) AS distinct_dates,
        COUNT(DISTINCT store_nbr) AS distinct_stores,
        COUNT(DISTINCT item_nbr) AS distinct_items,
        MIN(date) AS first_date,
        MAX(date) AS last_date
    FROM train
""").df()

,total_rows,distinct_dates,distinct_stores,distinct_items,first_date,last_date
0,125497040,1684,54,4036,2013-01-01,2017-08-15


In [142]:
con.execute("""
    CREATE OR REPLACE TABLE daily_store_family_sales AS
    SELECT
        t.date,
        t.store_nbr,
        i.family,
        SUM(t.unit_sales) AS daily_unit_sales,
        COUNT(*) AS product_records,
        COUNT(DISTINCT t.item_nbr) AS distinct_items
    FROM train AS t
    INNER JOIN items AS i
        ON t.item_nbr = i.item_nbr
    GROUP BY
        t.date,
        t.store_nbr,
        i.family
""")

In [143]:
con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT date) AS distinct_dates,
        COUNT(DISTINCT store_nbr) AS distinct_stores,
        COUNT(DISTINCT family) AS distinct_families,
        SUM(daily_unit_sales) AS total_unit_sales
    FROM daily_store_family_sales
""").df()

,total_rows,distinct_dates,distinct_stores,distinct_families,total_unit_sales
0,2061903,1684,54,33,1.073610e+09


In [144]:
con.execute("""
    SELECT
        'train' AS source,
        SUM(unit_sales) AS total_unit_sales
    FROM train

    UNION ALL

    SELECT
        'daily_store_family_sales' AS source,
        SUM(daily_unit_sales) AS total_unit_sales
    FROM daily_store_family_sales
""").df()

,source,total_unit_sales
0,train,1.073610e+09
1,daily_store_family_sales,1.073610e+09


In [145]:
con.execute("""
    CREATE OR REPLACE TABLE monthly_store_family_sales AS
    SELECT
        YEAR(date) AS year,
        MONTH(date) AS month,
        store_nbr,
        family,
        SUM(daily_unit_sales) AS total_unit_sales,
        COUNT(DISTINCT date) AS observed_days,
        COUNT(DISTINCT date) * 1.0
            / DAY(LAST_DAY(MIN(date))) AS observed_day_ratio
    FROM daily_store_family_sales
    GROUP BY
        YEAR(date),
        MONTH(date),
        store_nbr,
        family
""")

In [146]:
con.execute("""
    CREATE OR REPLACE TABLE store_item_sales AS
    SELECT
        store_nbr,
        item_nbr,
        SUM(unit_sales) AS total_unit_sales,
        COUNT(DISTINCT date) AS observed_days,
        COUNT(*) AS product_records,
        AVG(unit_sales) AS avg_unit_sales_per_record
    FROM train
    GROUP BY
        store_nbr,
        item_nbr
""")

In [147]:
con.execute("""
    CREATE OR REPLACE TABLE yearly_family_sales AS
    SELECT
        YEAR(t.date) AS year,
        i.family,
        SUM(t.unit_sales) AS total_unit_sales,
        COUNT(DISTINCT t.item_nbr) AS distinct_items,
        COUNT(DISTINCT t.store_nbr) AS distinct_stores
    FROM train AS t
    INNER JOIN items AS i
        ON t.item_nbr = i.item_nbr
    GROUP BY
        YEAR(t.date),
        i.family
""")

In [148]:
con.execute("""
    SELECT 'daily_store_family_sales' AS table_name,
           COUNT(*) AS row_count
    FROM daily_store_family_sales

    UNION ALL

    SELECT 'monthly_store_family_sales',
           COUNT(*)
    FROM monthly_store_family_sales

    UNION ALL

    SELECT 'store_item_sales',
           COUNT(*)
    FROM store_item_sales

    UNION ALL

    SELECT 'yearly_family_sales',
           COUNT(*)
    FROM yearly_family_sales
""").df()

,table_name,row_count
0,daily_store_family_sales,2061903
1,monthly_store_family_sales,76133
2,store_item_sales,174685
3,yearly_family_sales,152


In [149]:
from pathlib import Path

export_dir = Path(
    r"D:\Data Analysis Projects_\favorita-grocery-sales-forecasting\Notebook\favorita_exports\Parquet"
)
export_dir.mkdir(parents=True, exist_ok=True)

tables_to_export = [
    "daily_store_family_sales",
    "monthly_store_family_sales",
    "store_item_sales",
    "yearly_family_sales",
]

for table_name in tables_to_export:
    output_path = export_dir / f"{table_name}.parquet"

    con.execute(
        f"""
        COPY "{table_name}"
        TO '{output_path.as_posix()}'
        (FORMAT PARQUET, COMPRESSION ZSTD)
        """
    )

    print(f"Exported: {table_name}.parquet")

Exported: daily_store_family_sales.parquet
Exported: monthly_store_family_sales.parquet
Exported: store_item_sales.parquet
Exported: yearly_family_sales.parquet


In [150]:
from pathlib import Path

export_dir = Path(
    r"D:\Data Analysis Projects_\favorita-grocery-sales-forecasting\Notebook\favorita_exports\Parquet"
)

for file in sorted(export_dir.glob("*.parquet")):
    print(file.name)

daily_store_family_sales.parquet
daily_store_sales.parquet
items_pl.parquet
monthly_store_family_sales.parquet
store_item_sales.parquet
stores_pl.parquet
train_df.parquet
transactions_pl.parquet
yearly_family_sales.parquet


In [155]:
import duckdb

con = duckdb.connect()

file_path = r"D:\Data Analysis Projects\favorita-grocery-sales-forecasting\Notebook\favorita_exports\Parquet\train_df.parquet"

result = con.execute("""
    SELECT
        item_nbr,
        SUM(unit_sales) AS total_unit_sales
    FROM read_parquet(?)
    GROUP BY item_nbr
    ORDER BY total_unit_sales DESC
    LIMIT 10
""", [file_path]).fetchdf()

result

IOException: IO Error: No files found that match the pattern "D:\Data Analysis Projects\favorita-grocery-sales-forecasting\Notebook\favorita_exports\Parquet\train_df.parquet"

LINE 4:     FROM read_parquet(?)
                      ^^^^^^^^^^

In [157]:
file_path = r"D:\Data Analysis Projects_\favorita-grocery-sales-forecasting\Notebook\favorita_exports\Parquet\train_df.parquet"

result = con.execute("""
    SELECT
        item_nbr,
        SUM(unit_sales) AS total_unit_sales
    FROM read_parquet(?)
    GROUP BY item_nbr
    ORDER BY total_unit_sales DESC
    LIMIT 10
""", [file_path]).fetchdf()

result

,item_nbr,total_unit_sales
0,1503844,6.264200e+06
1,1047679,5.512875e+06
2,1473474,4.990323e+06
3,364606,4.416426e+06
4,819932,4.410943e+06
5,1463992,4.408545e+06
6,807493,4.321424e+06
7,265559,4.112827e+06
8,502331,3.703350e+06
9,314384,3.544341e+06


In [159]:
output_path = r"D:\Data Analysis Projects_\favorita-grocery-sales-forecasting\Notebook\favorita_exports\CSV\product_sales.csv"

con.execute(f"""
    COPY (
        SELECT
            item_nbr,
            SUM(unit_sales) AS total_unit_sales
        FROM read_parquet('{file_path}')
        GROUP BY item_nbr
    )
    TO '{output_path}' (HEADER, DELIMITER ',')
""")